# 실습 01 · 좌표 왕복

> **Day15 | 객체탐지·YOLO·추적 · 02_응용**
> 원본 스크립트: `01_coordinates.py` · 교안 PPT 15쪽 · 직접 구현: A01
> 코드: 완성 정답 기준(주석본 `02_응용_주석/01_coordinates.py`와 같은 로직) · 결과: 이 노트북을 실제 실행한 출력

## 학습 목표

- 픽셀 xyxy를 YOLO 정규화 cxcywh로 바꾸고 다시 되돌려 왕복 오차를 확인한다.
- x는 W, y는 H로 나눠야 하는 이유를 숫자로 확인한다.

## 핵심 개념

| 개념 | 설명 |
|---|---|
| xyxy | 왼쪽 위·오른쪽 아래 픽셀 |
| cxcywh 정규화 | 중심·크기를 W, H로 나눈 0~1 값 |
| 왕복 오차 | 변환 후 되돌린 좌표와 원본 차이 |

## 0. 준비

- 이 노트북은 `02_응용` 폴더(같은 폴더에 `data/`가 있는 위치)에서 연다.
- 입력은 `data/`의 원본을 읽기만 하고, 노트북이 만드는 결과 파일은 `outputs_notebook/`에 저장한다.
- CPU에서 실행한다. GPU·CUDA 코드는 없다.

In [1]:
# ============================================================
# 0. 라이브러리·경로 설정
# ============================================================
from pathlib import Path
import logging
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BASE : 노트북 폴더(02_응용) / DATA : 원본 입력 / OUT : 노트북 결과 폴더
BASE = Path.cwd()
DATA = BASE / "data"
OUT = BASE / "outputs_notebook"
OUT.mkdir(exist_ok=True)
# ultralytics 로그는 WARNING 이상만 출력
logging.getLogger("ultralytics").setLevel(logging.WARNING)

def show(images, titles, size=3.2):
    """BGR 배열 여러 장을 한 줄로 표시한다.

    - images: BGR 배열 목록
    - titles: 각 그림 제목 목록 (영문)
    """
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.9))
    axes = np.atleast_1d(axes)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(title, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

## 1. xyxy → 정규화 cxcywh (TODO A01)

In [2]:
# ============================================================
# 1. xyxy -> 정규화 cxcywh (TODO A01)
# ============================================================
table = pd.read_csv(DATA/"boxes_gt.csv")
b = table[["x1", "y1", "x2", "y2"]].to_numpy(float); width, height = 160, 128
# A01 : 중심·크기 계산 후 각 축 길이로 나눔
yolo = np.column_stack((
    (b[:, 0] + b[:, 2]) / 2 / width,
    (b[:, 1] + b[:, 3]) / 2 / height,
    (b[:, 2] - b[:, 0]) / width,
    (b[:, 3] - b[:, 1]) / height,
))
pd.DataFrame(yolo, columns=["cx", "cy", "w", "h"]).to_csv(OUT/"yolo_coordinates.csv", index=False)
pd.DataFrame(yolo, columns=["cx", "cy", "w", "h"])

,cx,cy,w,h
0,0.20000,0.296875,0.2500,0.312500
1,0.55625,0.617188,0.2375,0.296875
2,0.83125,0.312500,0.2125,0.343750


## 2. 왕복 오차

In [3]:
# ============================================================
# 2. 다시 픽셀로 되돌려 왕복 오차 확인
# ============================================================
pixel = yolo * np.array([width, height, width, height])
restored = np.column_stack((pixel[:, 0]-pixel[:, 2]/2, pixel[:, 1]-pixel[:, 3]/2, pixel[:, 0]+pixel[:, 2]/2, pixel[:, 1]+pixel[:, 3]/2))
print("최대 왕복 오차(px):", float(np.max(np.abs(b - restored))))

최대 왕복 오차(px): 0.0


## 3. 축을 바꿔 정규화하면

x를 H(128), y를 W(160)로 나눈 뒤 올바른 방식으로 되돌리면 좌표가 틀어진다.

In [4]:
# ============================================================
# 3. 축을 바꿔 정규화한 경우의 오차
# ============================================================
wrong = np.column_stack(((b[:, 0]+b[:, 2])/2/height, (b[:, 1]+b[:, 3])/2/width, (b[:, 2]-b[:, 0])/height, (b[:, 3]-b[:, 1])/width))
wp = wrong * np.array([width, height, width, height])
wrong_restored = np.column_stack((wp[:, 0]-wp[:, 2]/2, wp[:, 1]-wp[:, 3]/2, wp[:, 0]+wp[:, 2]/2, wp[:, 1]+wp[:, 3]/2))
print("축을 바꿨을 때 최대 오차(px):", round(float(np.max(np.abs(b - wrong_restored))), 2))
pd.DataFrame(np.round(wrong_restored, 1), columns=["x1", "y1", "x2", "y2"])

축을 바꿨을 때 최대 오차(px): 37.5


,x1,y1,x2,y2
0,15.0,14.4,65.0,46.4
1,87.5,48.0,135.0,78.4
2,145.0,14.4,187.5,49.6


## 정리 (README 설명 질문)

- **무엇을 고정했고 무엇을 바꾸었는가?** 입력 박스 3개와 이미지 크기 160×128을 고정하고, 변환 방향(xyxy→cxcywh→xyxy)만 적용했다. 비교용으로 축을 바꾼 잘못된 정규화도 따로 계산했다.
- **결과만으로 단정할 수 없는 부분은?** 왕복 오차 0.0은 이 3개 박스가 정수 좌표이고 경계 안에 있어서 나온 값이다. 경계 밖 박스, 실수 좌표의 반올림, 다른 해상도 이미지까지 안전하다는 증거는 아니다.